# 04 · What the paper's real 3D head would need

The paper images a 22 cm head on a 0.5 mm grid (440 cubes per side) with 1,024 transducers; its classical run took
~32 h on 128 CPU nodes, with a target of <10 min on GPUs. Here we scale our measured numbers up to that size.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # so `import qwave` works from notebooks/
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
RESULTS = pathlib.Path.cwd().parent / "results"; RESULTS.mkdir(exist_ok=True)
from qwave import Grid, head_phantom
from qwave.resources import paper_head, grid_counts, readout_shots, steps_needed, ibm_qpu_seconds
from qwave.encoding import unary_bonds
from qwave.circuits import unary_circuit, gate_cost
h = paper_head(); h

{'cells_per_side': 440,
 'vertices': 85184000,
 'edges': 256132800,
 'compact_qubits': 29,
 'unary_qubits': 341316800}

## 1 · Time steps grow with head size (the wave has to physically travel)

In [2]:
sizes, steps = [], []
for n in (3, 5, 7, 9, 11):
    g = Grid((n, n, n)); c, _ = head_phantom((n, n, n)); c = c.ravel()
    src = g.index(0, n // 2, n // 2); rx = [g.index(n - 1, n // 2, n // 2), g.index(n // 2, n // 2, n // 2)]
    s = steps_needed(g, c, t=float(n), source=src, receivers=rx, rel_tol=0.05)
    sizes.append(n); steps.append(s); print(f"{n}³ grid: {s} Trotter steps for 5% accuracy at crossing time")
print("Small grids are noisy (power-of-two steps, tiny signals), so we do NOT fit a growth law.")
print("Physical lower bound: the wave must cross the head, so steps grow at least linearly with n.")

3³ grid: 256 Trotter steps for 5% accuracy at crossing time


5³ grid: 64 Trotter steps for 5% accuracy at crossing time


7³ grid: 64 Trotter steps for 5% accuracy at crossing time


9³ grid: 128 Trotter steps for 5% accuracy at crossing time


11³ grid: 256 Trotter steps for 5% accuracy at crossing time
Small grids are noisy (power-of-two steps, tiny signals), so we do NOT fit a growth law.
Physical lower bound: the wave must cross the head, so steps grow at least linearly with n.


## 2 · Gates per step (unary), measured on a small 3D grid, then scaled

In [3]:
g = Grid((3, 3, 3)); c, _ = head_phantom((3, 3, 3)); c = c.ravel()
cz_per_step = gate_cost(unary_circuit(g, c, 1.0, 4, 0))["cz"] / 4
bonds = len(unary_bonds(g, c)); cz_per_bond = cz_per_step / bonds
print(f"CZ per bond per step ≈ {cz_per_bond:.2f}")
n = h["cells_per_side"]
bonds_head = 2 * h["edges"]                                     # each link touches ≤2 cubes
steps_lo = max(steps) * (n / sizes[-1])           # optimistic: linear growth from our largest measured grid
cz_lo = cz_per_bond * bonds_head * steps_lo
print(f"Real head, unary: {h['unary_qubits']:.2e} qubits, at least ~{steps_lo:.0e} steps, at least ~{cz_lo:.0e} CZ gates per simulation")
print(f"Real head, compact: {h['compact_qubits']} qubits (memory win), gate count needs fault-tolerant block-encoding methods;")
print("   our generic compact circuits already need ~10^6 CZ for a 3×3×3 head (notebook 02).")

CZ per bond per step ≈ 3.42
Real head, unary: 3.41e+08 qubits, at least ~1e+04 steps, at least ~2e+13 CZ gates per simulation
Real head, compact: 29 qubits (memory win), gate count needs fault-tolerant block-encoding methods;
   our generic compact circuits already need ~10^6 CZ for a 3×3×3 head (notebook 02).


## 3 · The read-out wall

In [4]:
r = readout_shots(sources=1024, receivers=1024, time_samples=1000, rel_precision=0.01, p_typical=1e-3)
secs = ibm_qpu_seconds(r["total_shots"], jobs=r["circuits"])
print(f"Numbers FWI needs per iteration: {r['numbers_needed']:.1e}")
print(f"Shots needed per iteration:      {r['total_shots']:.1e}")
print(f"At today's IBM rate:             {secs / 3.15e7:.0f} years per iteration")
print(f"Even at 1 MHz shot rate:         {r['total_shots'] / 1e6 / 86400:.0f} days per iteration")
print("Paper's GPU target:              < 10 minutes for the WHOLE reconstruction")

Numbers FWI needs per iteration: 1.0e+09
Shots needed per iteration:      1.0e+13
At today's IBM rate:             114 years per iteration
Even at 1 MHz shot rate:         118 days per iteration
Paper's GPU target:              < 10 minutes for the WHOLE reconstruction


**Takeaways**
* The compact encoding fits the real head in **29 qubits** — the genuine quantum promise (memory).
* Time steps grow with head size, at least linearly, just like the classical method: quantum saves memory, not steps.
  The 3D gate count is a lower-bound estimate (linear scaling from an 11³ grid), not a measurement.
* Reading the full recordings out costs ~10¹³ shots per iteration — hopeless even for perfect future hardware.
  Quantum only makes sense for questions with a **single-number answer** (see notebook 05).